# Lasso Regression — Medical Insurance Charges

Extends the linear regression baseline by adding L1 regularisation (Lasso) and feature engineering.

**Why Lasso over plain linear regression?**  
Lasso adds a penalty proportional to the sum of absolute coefficient values. This shrinks less-important feature weights toward zero, effectively performing feature selection. It is especially useful when the feature set contains redundant or weakly predictive columns.

**Additions over the baseline notebook:**
- One-hot encoding for the `region` column (`pd.get_dummies`)
- Two interaction terms: `age × smoker` and `bmi × smoker` — these capture the non-linear amplification effect smoking has on charges for older/heavier patients
- Manual alpha sweep to visualise the MSE–alpha trade-off
- `LassoCV` with 5-fold cross-validation to select the optimal regularisation strength automatically

**Key result:** R² ≈ 0.865 — a meaningful improvement over the plain linear model (~0.78), driven by the interaction terms rather than regularisation alone.

In [2]:
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
# One-hot encode the region column.
# drop_first=True removes one dummy column per group to avoid perfect multicollinearity
# (the "dummy variable trap") — e.g. if southwest/southeast/northwest are all 0, it must be northeast.

insurance_data = pd.read_csv("data-medical.csv")

X = insurance_data.drop(columns=["charges"])
y = insurance_data["charges"]

X = pd.get_dummies(X, columns=["region"], drop_first=True, dtype=int)

X["sex"] = X["sex"].map({"female": 1, "male": 0})
X["smoker"] = X["smoker"].map({"yes": 1, "no": 0})

# Interaction terms — smoking dramatically amplifies the effect of age and BMI on charges.
# Multiplying creates a new feature that captures this joint effect explicitly.
X["age_smoker"] = X["age"] * X["smoker"]
X["bmi_smoker"] = X["bmi"] * X["smoker"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
import seaborn as sns

# Manually sweep over a range of alpha values to understand how regularisation strength
# affects test-set error. Higher alpha → stronger penalty → coefficients shrink more →
# model becomes simpler but potentially underfits.
alphas = [0.001, 0.1, 1, 2, 5, 10, 20, 30, 40, 50, 100]
mses = []

for a in alphas:
    lasso_model = Lasso(alpha=a)
    lasso_model.fit(X_train, y_train)
    
    y_pred = lasso_model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    print(f"MSE for alpha={a}: ", mse)
    mses.append(mse)

# Plot the MSE curve — the elbow point shows the best balance between bias and variance.
sns.lineplot(x=alphas, y=mses, marker="o")

In [ ]:
from sklearn.linear_model import LassoCV

a = [0.001, 0.1, 1, 2, 5, 10, 20, 30, 40, 50, 100]

# LassoCV automates alpha selection using k-fold cross-validation.
# cv=5: each alpha is evaluated by training on 4 folds and validating on the 5th, repeated 5 times.
# The alpha with the lowest average validation MSE across all folds is selected.
lasso_cv_model = LassoCV(
    alphas=a,
    cv=5,
    max_iter=1000,
    random_state=42
)

lasso_cv_model.fit(X_train, y_train)

print("best alpha: ", lasso_cv_model.alpha_)  # the alpha chosen by cross-validation

y_pred = lasso_cv_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print("mse = ", mse)
print("r2 = ", r2)